In [272]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/datasets', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [273]:
import pandas as pd
import glob
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.utils import resample
import tensorflow as tf
import csv
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, TextVectorization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

In [274]:
all_files = glob.glob("/content/drive/MyDrive/datasets/*.csv")

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {[os.path.basename(f) for f in all_files]}")
df.info()

Total rows: 8236
Dari 2 file: ['Linkedin_finalfix_1105.csv', 'Jobstreet Cleaned.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8236 entries, 0 to 8235
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                8236 non-null   float64
 1   title             8236 non-null   object 
 2   search_role       8236 non-null   object 
 3   job_description   8231 non-null   object 
 4   extracted_skills  8236 non-null   object 
 5   skills_count      8236 non-null   int64  
 6   search_role_raw   5902 non-null   object 
 7   job_level         5902 non-null   object 
 8   company           5902 non-null   object 
 9   location          5902 non-null   object 
 10  location_raw      5902 non-null   object 
 11  salary_display    5902 non-null   object 
 12  salary_min        1667 non-null   float64
 13  salary_max        1667 non-null   float64
 14  salary_avg        1667 non-null   float64
 15  job

In [275]:
df = df.drop_duplicates()
df = df.drop(columns=['salary', 'search_role_raw', 'job_level', 'location_raw',
                       'salary_display', 'salary_min', 'salary_max', 'salary_avg', 'company', 'location', 'job_url', 'search_location', 'scraped_at'],
             errors='ignore')
print(f"Total duplicated rows: {df.duplicated().sum()}")
df.dropna(inplace=True)
print(df.isnull().sum())

Total duplicated rows: 0
id                  0
title               0
search_role         0
job_description     0
extracted_skills    0
skills_count        0
dtype: int64


In [276]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8076 entries, 0 to 8235
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                8076 non-null   float64
 1   title             8076 non-null   object 
 2   search_role       8076 non-null   object 
 3   job_description   8076 non-null   object 
 4   extracted_skills  8076 non-null   object 
 5   skills_count      8076 non-null   int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 441.7+ KB


In [277]:
def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)
print(f"Total rows: {len(df)}")
print(f"Skills kosong: {(df['extracted_skills_clean'] == '').sum()} rows")
print(df['search_role'].value_counts())

Total rows: 8076
Skills kosong: 437 rows
search_role
Backend Developer      1365
Fullstack Developer    1276
DevOps Engineer        1254
Data Analyst           1032
Frontend Developer     1024
Data Engineer           963
Software Engineer       478
ML Engineer             381
Data Scientist          303
Name: count, dtype: int64


In [278]:
min_samples = 900
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Backend Developer      1365
Fullstack Developer    1276
DevOps Engineer        1254
Data Analyst           1032
Frontend Developer     1024
Data Engineer           963
Name: count, dtype: int64


In [279]:
df['search_role'] = df['search_role'].replace({'Full stack Developer': 'Fullstack Developer'})

roles_to_drop = ['Software Engineer', 'Web Developer', 'IT Support', 'IT',
                 'Developer', 'Software', 'System Analyst', 'Progammer']
df = df[~df['search_role'].isin(roles_to_drop)]

min_samples = 200
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)

print(df['search_role'].value_counts())

search_role
Backend Developer      1365
Fullstack Developer    1276
DevOps Engineer        1254
Data Analyst           1032
Frontend Developer     1024
Data Engineer           963
Name: count, dtype: int64


In [280]:
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])
num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")
print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

target_samples = 1200
dfs_resampled = []
for role in df['search_role'].unique():
    df_role = df[df['search_role'] == role]
    if len(df_role) < target_samples:
        df_role = resample(df_role, replace=True, n_samples=target_samples, random_state=42)
    else:
        df_role = df_role.sample(n=target_samples, random_state=42)
    dfs_resampled.append(df_role)

df = pd.concat(dfs_resampled).sample(frac=1, random_state=42).reset_index(drop=True)
df['label'] = encoder.transform(df['search_role'])
print(df['search_role'].value_counts())

df['text_input'] = df['title'] + ' ' + df['job_description'] + ' ' + df['extracted_skills_clean']
df['text_input'] = df['text_input'].fillna('').astype(str)

Total role unik: 6
Mapping kelas:
{'Backend Developer': 0, 'Data Analyst': 1, 'Data Engineer': 2, 'DevOps Engineer': 3, 'Frontend Developer': 4, 'Fullstack Developer': 5}
search_role
Backend Developer      1200
Frontend Developer     1200
Data Engineer          1200
Data Analyst           1200
DevOps Engineer        1200
Fullstack Developer    1200
Name: count, dtype: int64


In [281]:
print("=== Label alignment check ===")
for i in range(5):
    print(f"Role: {df['search_role'].iloc[i]}")
    print(f"Label: {df['label'].iloc[i]}")
    print(f"Title: {df['title'].iloc[i]}")
    print()

=== Label alignment check ===
Role: Backend Developer
Label: 0
Title: Team Lead Developer

Role: Backend Developer
Label: 0
Title: Back-end Developer (Android)

Role: Frontend Developer
Label: 4
Title: Software Developer - Banking Industry

Role: Data Engineer
Label: 2
Title: Manager Raw Material

Role: Backend Developer
Label: 0
Title: Full Stack Developer



In [282]:
X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

vectorizer = TextVectorization(
    max_tokens=20000,
    output_mode='int',
    output_sequence_length=512
)
vectorizer.adapt(X_train.to_numpy())
print("Shape X_train:", X_train.shape)

Shape X_train: (5760,)


In [283]:
print("=== Sample text inputs ===")
for role in df['search_role'].unique():
    sample = df[df['search_role'] == role]['text_input'].iloc[0]
    print(f"\n[{role}]")
    print(sample[:200])

print("\n=== Vectorizer vocab sample ===")
print(vectorizer.get_vocabulary()[:50])
print(f"\nVocab size: {len(vectorizer.get_vocabulary())}")

=== Sample text inputs ===

[Backend Developer]
Team Lead Developer Xtremax values developers with raw instincts in programming and the determination to scale Alpine mountains, not hike small hills. We look for talents who are up for the challenge 

[Frontend Developer]
Software Developer - Banking Industry Client: Banking Industry

Location: Bintaro Sektor 7 Tangerang Selatan (Full WFO)

Requirements:

1. Paham mengenai transaksi web or mobile banking

2. Menguasai 

[Data Engineer]
Manager Raw Material Job Qualification

Pendidikan minimal D3/S1 Teknik Mesin, Teknik Industri, Teknik Elektro atau jurusan terkait

Memiliki pengalaman minimal 2–3 tahun di bidang manufaktur, berpeng

[Data Analyst]
IT Project Manager Job Description

Mengelola pelaksanaan proyek secara end-to-end agar selesai tepat waktu, sesuai scope, dan sesuai anggaran.
Menyusun rencana proyek secara detail mencakup scope, ti

[DevOps Engineer]
Site Reliability Engineer / DevOps About the job

As a Site Reliability En

In [284]:
class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=5):
        super().__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CustomEarlyStopping(patience=5)
]

In [285]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=64)(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier")
model.compile(
    optimizer=Adam(learning_rate=0.002),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

print("Mulai proses training...")
history = model.fit(
    X_train.to_numpy(), y_train,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=40,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

Mulai proses training...
Epoch 1/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.2092 - loss: 1.7663 - val_accuracy: 0.3049 - val_loss: 1.6706 - learning_rate: 0.0020
Epoch 2/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3444 - loss: 1.5092 - val_accuracy: 0.4458 - val_loss: 1.3347 - learning_rate: 0.0020
Epoch 3/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.4668 - loss: 1.2390 - val_accuracy: 0.5326 - val_loss: 1.1411 - learning_rate: 0.0020
Epoch 4/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5467 - loss: 1.0823 - val_accuracy: 0.5625 - val_loss: 1.0465 - learning_rate: 0.0020
Epoch 5/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5944 - loss: 0.9577 - val_accuracy: 0.5701 - val_loss: 1.0202 - learning_rate: 0.0020
Epoch 6/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6160 - loss: 0.9031 - val_accuracy: 0.5583 - val_loss: 1.0355 - learning_rate: 0.0020
Epoch 7/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6484

In [286]:
y_pred = np.argmax(model.predict(X_val.to_numpy()), axis=1)
print(classification_report(y_val, y_pred, target_names=encoder.classes_))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
                     precision    recall  f1-score   support

  Backend Developer       0.43      0.33      0.37       240
       Data Analyst       0.79      0.79      0.79       240
      Data Engineer       0.90      0.88      0.89       240
    DevOps Engineer       0.63      0.64      0.63       240
 Frontend Developer       0.52      0.62      0.57       240
Fullstack Developer       0.58      0.61      0.60       240

           accuracy                           0.65      1440
          macro avg       0.64      0.65      0.64      1440
       weighted avg       0.64      0.65      0.64      1440



In [287]:
model.save('job_role_model.keras')
print("[INFO] Model berhasil disimpan")

loaded_model = tf.keras.models.load_model('job_role_model.keras')

def rekomendasi_role(teks_skill_user):
    input_tensor = tf.constant([teks_skill_user], dtype=tf.string)
    pred_probs = loaded_model.predict(input_tensor, verbose=0)
    pred_class_idx = np.argmax(pred_probs)
    predicted_role = encoder.inverse_transform([pred_class_idx])[0]
    return predicted_role

skill_input = "i have experience for creating web interface using reactjs, next.js, tailwind, and using operating system linux ubuntu"
hasil_prediksi = rekomendasi_role(skill_input)

print("\n--- Hasil Inference ---")
print(f"Skill User : {skill_input}")
print(f"Cocok untuk: {hasil_prediksi}")

[INFO] Model berhasil disimpan

--- Hasil Inference ---
Skill User : i have experience for creating web interface using reactjs, next.js, tailwind, and using operating system linux ubuntu
Cocok untuk: Backend Developer
